# Comprehensive KAN Variant Tuning

Hyperparameter optimization for ALL KAN variants:
- **ChebyKAN** (Chebyshev polynomials)
- **FourierKAN** (Fourier series)
- **FastKAN** (Radial Basis Functions)
- **Wav-KAN** (Wavelets)
- **efficient-kan** (B-splines)

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import time
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
import xgboost as xgb

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

np.random.seed(42)
torch.manual_seed(42)
device = torch.device('mps' if torch.backends.mps.is_available() else 'cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

# Load data
df = pd.read_excel('../data/High_Accuracy_Sport_Injury_Dataset.xlsx')
X = df.drop('Injury_Risk', axis=1).values
y = df['Injury_Risk'].values
feature_names = df.drop('Injury_Risk', axis=1).columns.tolist()
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
n_features = X.shape[1]

X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42, stratify=y)
print(f"Train: {len(X_train)}, Test: {len(X_test)}, Features: {n_features}")

Device: mps


Train: 480, Test: 120, Features: 15


## KAN Implementations

In [2]:
# Base MLP for comparison
class MLP(nn.Module):
    def __init__(self, dims):
        super().__init__()
        layers = []
        for i in range(len(dims)-1):
            layers.append(nn.Linear(dims[i], dims[i+1]))
            if i < len(dims)-2:
                layers.extend([nn.BatchNorm1d(dims[i+1]), nn.ReLU(), nn.Dropout(0.2)])
        layers.append(nn.Sigmoid())
        self.net = nn.Sequential(*layers)
    def forward(self, x): return self.net(x)

# ChebyKAN
class ChebyKANLayer(nn.Module):
    def __init__(self, in_f, out_f, degree=4):
        super().__init__()
        self.degree = degree
        self.coeffs = nn.Parameter(torch.randn(in_f, out_f, degree + 1) * 0.1)
        self.bias = nn.Parameter(torch.zeros(out_f))
    def forward(self, x):
        x_n = torch.tanh(x)
        T = [torch.ones_like(x_n), x_n]
        for _ in range(2, self.degree + 1): T.append(2 * x_n * T[-1] - T[-2])
        return torch.einsum('bid,iod->bo', torch.stack(T, dim=-1), self.coeffs) + self.bias

class ChebyKAN(nn.Module):
    def __init__(self, dims, degree=4):
        super().__init__()
        self.layers = nn.ModuleList([ChebyKANLayer(dims[i], dims[i+1], degree) for i in range(len(dims)-1)])
    def forward(self, x):
        for l in self.layers: x = l(x)
        return torch.sigmoid(x)

# FourierKAN
class FourierKANLayer(nn.Module):
    def __init__(self, in_f, out_f, grid_size=8):
        super().__init__()
        self.grid_size = grid_size
        self.a = nn.Parameter(torch.randn(in_f, out_f, grid_size) * 0.1)
        self.b = nn.Parameter(torch.randn(in_f, out_f, grid_size) * 0.1)
        self.bias = nn.Parameter(torch.zeros(out_f))
        self.freq_scale = nn.Parameter(torch.ones(1))
    def forward(self, x):
        freqs = torch.arange(1, self.grid_size + 1, device=x.device).float() * self.freq_scale
        x_exp = x.unsqueeze(-1)
        cos_t = torch.cos(freqs * x_exp * np.pi)
        sin_t = torch.sin(freqs * x_exp * np.pi)
        return torch.einsum('big,iog->bo', cos_t, self.a) + torch.einsum('big,iog->bo', sin_t, self.b) + self.bias

class FourierKAN(nn.Module):
    def __init__(self, dims, grid_size=8):
        super().__init__()
        self.layers = nn.ModuleList([FourierKANLayer(dims[i], dims[i+1], grid_size) for i in range(len(dims)-1)])
    def forward(self, x):
        for l in self.layers: x = l(x)
        return torch.sigmoid(x)

# FastKAN (RBF)
class RBFKANLayer(nn.Module):
    def __init__(self, in_f, out_f, num_centers=16):
        super().__init__()
        self.centers = nn.Parameter(torch.linspace(-2, 2, num_centers).unsqueeze(0).unsqueeze(0).repeat(in_f, out_f, 1))
        self.log_bw = nn.Parameter(torch.zeros(in_f, out_f, num_centers))
        self.weights = nn.Parameter(torch.randn(in_f, out_f, num_centers) * 0.1)
        self.bias = nn.Parameter(torch.zeros(out_f))
    def forward(self, x):
        x_exp = x.unsqueeze(2).unsqueeze(3)
        bw = torch.exp(self.log_bw) + 0.1
        rbf = torch.exp(-((x_exp - self.centers) ** 2) / (2 * bw ** 2))
        return (rbf * self.weights).sum(dim=-1).sum(dim=1) + self.bias

class FastKAN(nn.Module):
    def __init__(self, dims, num_centers=16):
        super().__init__()
        self.layers = nn.ModuleList([RBFKANLayer(dims[i], dims[i+1], num_centers) for i in range(len(dims)-1)])
    def forward(self, x):
        for l in self.layers: x = l(x)
        return torch.sigmoid(x)

# WavKAN (Wavelets)
class WaveletKANLayer(nn.Module):
    def __init__(self, in_f, out_f, num_wavelets=16):
        super().__init__()
        self.trans = nn.Parameter(torch.linspace(-3, 3, num_wavelets).unsqueeze(0).unsqueeze(0).repeat(in_f, out_f, 1))
        self.log_scale = nn.Parameter(torch.zeros(in_f, out_f, num_wavelets))
        self.weights = nn.Parameter(torch.randn(in_f, out_f, num_wavelets) * 0.1)
        self.bias = nn.Parameter(torch.zeros(out_f))
    def mexican_hat(self, x): return (1 - x**2) * torch.exp(-x**2 / 2)
    def forward(self, x):
        x_exp = x.unsqueeze(2).unsqueeze(3)
        scale = torch.exp(self.log_scale) + 0.1
        x_norm = (x_exp - self.trans) / scale
        return (self.mexican_hat(x_norm) * self.weights).sum(dim=-1).sum(dim=1) + self.bias

class WavKAN(nn.Module):
    def __init__(self, dims, num_wavelets=16):
        super().__init__()
        self.layers = nn.ModuleList([WaveletKANLayer(dims[i], dims[i+1], num_wavelets) for i in range(len(dims)-1)])
    def forward(self, x):
        for l in self.layers: x = l(x)
        return torch.sigmoid(x)

## Training Function

In [3]:
def train_and_eval(model, X_train, y_train, X_test, y_test, epochs=200, lr=0.003, batch_size=32):
    model = model.to(device)
    X_t = torch.FloatTensor(X_train)
    y_t = torch.FloatTensor(y_train).unsqueeze(1)
    X_test_t = torch.FloatTensor(X_test)
    
    loader = DataLoader(TensorDataset(X_t, y_t), batch_size=batch_size, shuffle=True)
    opt = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    crit = nn.BCELoss()
    
    start = time.time()
    for _ in range(epochs):
        model.train()
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            opt.zero_grad()
            loss = crit(model(xb), yb)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
        scheduler.step()
    train_time = time.time() - start
    
    model.eval()
    with torch.no_grad():
        y_prob = model(X_test_t.to(device)).cpu().numpy().flatten()
    y_pred = (y_prob > 0.5).astype(int)
    
    return {
        'acc': accuracy_score(y_test, y_pred),
        'f1': f1_score(y_test, y_pred),
        'auc': roc_auc_score(y_test, y_prob),
        'time': train_time,
        'params': sum(p.numel() for p in model.parameters())
    }

## Hyperparameter Sweep for Each KAN Variant

In [4]:
print("=" * 70)
print("HYPERPARAMETER SWEEP FOR ALL KAN VARIANTS")
print("=" * 70)

all_results = []

# Baselines
print("\n--- BASELINES ---")
# XGBoost
start = time.time()
xgb_model = xgb.XGBClassifier(n_estimators=100, max_depth=6, random_state=42, eval_metric='logloss')
xgb_model.fit(X_train, y_train)
y_pred_xgb = xgb_model.predict(X_test)
y_prob_xgb = xgb_model.predict_proba(X_test)[:, 1]
all_results.append({
    'Model': 'XGBoost', 'Config': 'Default', 'Acc': accuracy_score(y_test, y_pred_xgb),
    'F1': f1_score(y_test, y_pred_xgb), 'AUC': roc_auc_score(y_test, y_prob_xgb),
    'Time': time.time() - start, 'Params': 'N/A'
})
print(f"XGBoost: Acc={all_results[-1]['Acc']:.4f}")

# MLP
for arch in [[n_features, 64, 32, 1], [n_features, 128, 64, 1]]:
    res = train_and_eval(MLP(arch), X_train, y_train, X_test, y_test, epochs=200)
    all_results.append({'Model': 'MLP', 'Config': str(arch), 'Acc': res['acc'], 'F1': res['f1'], 
                        'AUC': res['auc'], 'Time': res['time'], 'Params': res['params']})
    print(f"MLP {arch}: Acc={res['acc']:.4f}")

HYPERPARAMETER SWEEP FOR ALL KAN VARIANTS

--- BASELINES ---
XGBoost: Acc=0.9583


MLP [15, 64, 32, 1]: Acc=0.8500


MLP [15, 128, 64, 1]: Acc=0.8333


In [5]:
# ChebyKAN sweep
print("\n--- ChebyKAN ---")
cheby_configs = [
    ([n_features, 64, 32, 1], 3, 0.003),
    ([n_features, 64, 32, 1], 4, 0.003),
    ([n_features, 64, 32, 1], 5, 0.002),
    ([n_features, 100, 50, 1], 3, 0.002),
    ([n_features, 100, 50, 1], 4, 0.002),
]
for arch, deg, lr in cheby_configs:
    res = train_and_eval(ChebyKAN(arch, degree=deg), X_train, y_train, X_test, y_test, lr=lr)
    config = f"arch={arch}, deg={deg}"
    all_results.append({'Model': 'ChebyKAN', 'Config': config, 'Acc': res['acc'], 'F1': res['f1'],
                        'AUC': res['auc'], 'Time': res['time'], 'Params': res['params']})
    print(f"ChebyKAN {config}: Acc={res['acc']:.4f}")


--- ChebyKAN ---


ChebyKAN arch=[15, 64, 32, 1], deg=3: Acc=0.8333


ChebyKAN arch=[15, 64, 32, 1], deg=4: Acc=0.8667


ChebyKAN arch=[15, 64, 32, 1], deg=5: Acc=0.7583


ChebyKAN arch=[15, 100, 50, 1], deg=3: Acc=0.8667


ChebyKAN arch=[15, 100, 50, 1], deg=4: Acc=0.8417


In [6]:
# FourierKAN sweep
print("\n--- FourierKAN ---")
fourier_configs = [
    ([n_features, 64, 32, 1], 4, 0.005),
    ([n_features, 64, 32, 1], 8, 0.005),
    ([n_features, 64, 32, 1], 12, 0.003),
    ([n_features, 100, 50, 1], 8, 0.003),
]
for arch, grid, lr in fourier_configs:
    res = train_and_eval(FourierKAN(arch, grid_size=grid), X_train, y_train, X_test, y_test, lr=lr)
    config = f"arch={arch}, grid={grid}"
    all_results.append({'Model': 'FourierKAN', 'Config': config, 'Acc': res['acc'], 'F1': res['f1'],
                        'AUC': res['auc'], 'Time': res['time'], 'Params': res['params']})
    print(f"FourierKAN {config}: Acc={res['acc']:.4f}")


--- FourierKAN ---


FourierKAN arch=[15, 64, 32, 1], grid=4: Acc=0.5500


FourierKAN arch=[15, 64, 32, 1], grid=8: Acc=0.6833


FourierKAN arch=[15, 64, 32, 1], grid=12: Acc=0.6833


FourierKAN arch=[15, 100, 50, 1], grid=8: Acc=0.6917


In [7]:
# FastKAN (RBF) sweep
print("\n--- FastKAN (RBF) ---")
fast_configs = [
    ([n_features, 48, 24, 1], 8, 0.005),
    ([n_features, 48, 24, 1], 16, 0.003),
    ([n_features, 64, 32, 1], 16, 0.003),
    ([n_features, 64, 32, 1], 24, 0.002),
]
for arch, centers, lr in fast_configs:
    res = train_and_eval(FastKAN(arch, num_centers=centers), X_train, y_train, X_test, y_test, lr=lr)
    config = f"arch={arch}, centers={centers}"
    all_results.append({'Model': 'FastKAN', 'Config': config, 'Acc': res['acc'], 'F1': res['f1'],
                        'AUC': res['auc'], 'Time': res['time'], 'Params': res['params']})
    print(f"FastKAN {config}: Acc={res['acc']:.4f}")


--- FastKAN (RBF) ---


FastKAN arch=[15, 48, 24, 1], centers=8: Acc=0.9000


FastKAN arch=[15, 48, 24, 1], centers=16: Acc=0.8833


FastKAN arch=[15, 64, 32, 1], centers=16: Acc=0.9083


FastKAN arch=[15, 64, 32, 1], centers=24: Acc=0.9000


In [8]:
# WavKAN sweep
print("\n--- WavKAN (Wavelets) ---")
wav_configs = [
    ([n_features, 48, 24, 1], 8, 0.005),
    ([n_features, 48, 24, 1], 16, 0.003),
    ([n_features, 64, 32, 1], 16, 0.003),
    ([n_features, 64, 32, 1], 24, 0.002),
]
for arch, wavelets, lr in wav_configs:
    res = train_and_eval(WavKAN(arch, num_wavelets=wavelets), X_train, y_train, X_test, y_test, lr=lr)
    config = f"arch={arch}, wavelets={wavelets}"
    all_results.append({'Model': 'WavKAN', 'Config': config, 'Acc': res['acc'], 'F1': res['f1'],
                        'AUC': res['auc'], 'Time': res['time'], 'Params': res['params']})
    print(f"WavKAN {config}: Acc={res['acc']:.4f}")


--- WavKAN (Wavelets) ---


WavKAN arch=[15, 48, 24, 1], wavelets=8: Acc=0.9000


WavKAN arch=[15, 48, 24, 1], wavelets=16: Acc=0.8000


WavKAN arch=[15, 64, 32, 1], wavelets=16: Acc=0.8167


WavKAN arch=[15, 64, 32, 1], wavelets=24: Acc=0.8250


In [9]:
# efficient-kan
print("\n--- efficient-kan (B-splines) ---")
try:
    from efficient_kan import KAN as EfficientKAN
    class EKANClassifier(nn.Module):
        def __init__(self, layers, grid=5, k=3):
            super().__init__()
            self.kan = EfficientKAN(layers, grid_size=grid, spline_order=k)
        def forward(self, x): return torch.sigmoid(self.kan(x))
    
    ekan_configs = [
        ([n_features, 32, 16, 1], 5, 3, 0.005),
        ([n_features, 64, 32, 1], 5, 3, 0.003),
        ([n_features, 64, 32, 1], 8, 3, 0.003),
    ]
    for arch, grid, k, lr in ekan_configs:
        res = train_and_eval(EKANClassifier(arch, grid, k), X_train, y_train, X_test, y_test, lr=lr)
        config = f"arch={arch}, grid={grid}"
        all_results.append({'Model': 'efficient-kan', 'Config': config, 'Acc': res['acc'], 'F1': res['f1'],
                            'AUC': res['auc'], 'Time': res['time'], 'Params': res['params']})
        print(f"efficient-kan {config}: Acc={res['acc']:.4f}")
except Exception as e:
    print(f"efficient-kan failed: {e}")


--- efficient-kan (B-splines) ---


efficient-kan arch=[15, 32, 16, 1], grid=5: Acc=0.8833


efficient-kan arch=[15, 64, 32, 1], grid=5: Acc=0.8750


efficient-kan arch=[15, 64, 32, 1], grid=8: Acc=0.8833


## Results Summary

In [10]:
results_df = pd.DataFrame(all_results)
results_df = results_df.sort_values('Acc', ascending=False)

print("\n" + "=" * 70)
print("ALL RESULTS SORTED BY ACCURACY")
print("=" * 70)
display(results_df.round(4))

# Best per model type
print("\n" + "=" * 70)
print("BEST CONFIGURATION PER MODEL TYPE")
print("=" * 70)
best_per_model = results_df.loc[results_df.groupby('Model')['Acc'].idxmax()]
best_per_model = best_per_model.sort_values('Acc', ascending=False)
display(best_per_model.round(4))


ALL RESULTS SORTED BY ACCURACY


,Model,Config,Acc,F1,AUC,Time,Params
0,XGBoost,Default,0.9583,0.9296,0.9403,0.1223,N/A
14,FastKAN,"arch=[15, 64, 32, 1], centers=16",0.9083,0.8406,0.8992,15.9601,146017
12,FastKAN,"arch=[15, 48, 24, 1], centers=8",0.9000,0.8286,0.9162,13.4587,45577
16,WavKAN,"arch=[15, 48, 24, 1], wavelets=8",0.9000,0.8182,0.9397,14.7469,45577
15,FastKAN,"arch=[15, 64, 32, 1], centers=24",0.9000,0.8333,0.8973,19.1705,218977
20,efficient-kan,"arch=[15, 32, 16, 1], grid=5",0.8833,0.7941,0.8992,17.1358,10080
13,FastKAN,"arch=[15, 48, 24, 1], centers=16",0.8833,0.7941,0.9002,14.7880,91081
22,efficient-kan,"arch=[15, 64, 32, 1], grid=8",0.8833,0.8056,0.9345,15.8915,39520
21,efficient-kan,"arch=[15, 64, 32, 1], grid=5",0.8750,0.7826,0.9262,17.0684,30400
6,ChebyKAN,"arch=[15, 100, 50, 1], deg=3",0.8667,0.7576,0.8780,12.8155,26351



BEST CONFIGURATION PER MODEL TYPE


,Model,Config,Acc,F1,AUC,Time,Params
0,XGBoost,Default,0.9583,0.9296,0.9403,0.1223,N/A
14,FastKAN,"arch=[15, 64, 32, 1], centers=16",0.9083,0.8406,0.8992,15.9601,146017
16,WavKAN,"arch=[15, 48, 24, 1], wavelets=8",0.9000,0.8182,0.9397,14.7469,45577
20,efficient-kan,"arch=[15, 32, 16, 1], grid=5",0.8833,0.7941,0.8992,17.1358,10080
6,ChebyKAN,"arch=[15, 100, 50, 1], deg=3",0.8667,0.7576,0.8780,12.8155,26351
1,MLP,"[15, 64, 32, 1]",0.8500,0.7500,0.9002,13.5139,3329
11,FourierKAN,"arch=[15, 100, 50, 1], grid=8",0.6917,0.0513,0.4804,18.2002,104954


In [11]:
# Save best configs for final evaluation
best_configs = {
    'ChebyKAN': best_per_model[best_per_model['Model']=='ChebyKAN']['Config'].values[0] if 'ChebyKAN' in best_per_model['Model'].values else None,
    'FourierKAN': best_per_model[best_per_model['Model']=='FourierKAN']['Config'].values[0] if 'FourierKAN' in best_per_model['Model'].values else None,
    'FastKAN': best_per_model[best_per_model['Model']=='FastKAN']['Config'].values[0] if 'FastKAN' in best_per_model['Model'].values else None,
    'WavKAN': best_per_model[best_per_model['Model']=='WavKAN']['Config'].values[0] if 'WavKAN' in best_per_model['Model'].values else None,
}
print("\nBest configurations for final evaluation:")
for k, v in best_configs.items():
    print(f"  {k}: {v}")


Best configurations for final evaluation:
  ChebyKAN: arch=[15, 100, 50, 1], deg=3
  FourierKAN: arch=[15, 100, 50, 1], grid=8
  FastKAN: arch=[15, 64, 32, 1], centers=16
  WavKAN: arch=[15, 48, 24, 1], wavelets=8
